# LegalMitra: LoRA Adapter Merge & 4-bit GGUF Export

**Disk-Optimized Direct Quantization Pipeline:**
1. **Zero-Disk-Overflow Direct GGUF:** Uses modern `unsloth` / `llama.cpp` direct conversion: we merge LoRA and convert directly into 4-bit `q4_k_m` GGUF (~4.6 GB) without writing an uncompressed 16.1 GB FP16 intermediate file to disk.
2. **Cleans `/tmp` and Cache:** Frees up all 19.5 GB of Kaggle scratch space before quantization.
3. **Outputs `legalmitra-8b-q4_k_m.gguf` (~4.6 GB)** ready for offline local CPU inference.

**Hardware Setting:** Kaggle Free GPU (T4 x2 or P100) | **Internet:** Enabled

## Step 1: Install Dependencies, Remove Incompatible Packages & Build llama.cpp

In [ ]:
# 1. Clean environment
!pip uninstall -y torchao
!pip install -q "transformers>=4.44.0" "peft>=0.12.0" "accelerate>=0.33.0" sentencepiece huggingface_hub gguf

# 2. Clone llama.cpp and build conversion & quantization binaries
!rm -rf /kaggle/working/llama.cpp
!git clone --depth 1 https://github.com/ggerganov/llama.cpp.git /kaggle/working/llama.cpp
!cmake -B /kaggle/working/llama.cpp/build -S /kaggle/working/llama.cpp -DGGML_CUDA=OFF
!cmake --build /kaggle/working/llama.cpp/build --config Release -j4 --target llama-quantize llama-cli
!pip install -q -r /kaggle/working/llama.cpp/requirements.txt

print("SUCCESS: Environment configured and llama.cpp built cleanly.")

## Step 2: Locate the Uploaded LoRA Adapter Dataset

In [ ]:
import os
import glob

# Locate the adapter folder automatically across Kaggle input directories
adapter_candidates = glob.glob('/kaggle/input/**/adapter_model.safetensors', recursive=True)

if adapter_candidates:
    adapter_dir = os.path.dirname(adapter_candidates[0])
    print(f"Found LegalMitra LoRA adapter at: {adapter_dir}")
else:
    adapter_dir = '/kaggle/working/legalmitra_llama3_lora_final'
    if not os.path.exists(adapter_dir):
        raise FileNotFoundError(
            "LoRA adapter not found! Please attach your 'legalmitra-lora-adapter' Kaggle dataset "
            "via '+ Add Input' on the right panel."
        )
    print(f"Using local session adapter at: {adapter_dir}")

print("Adapter directory files:", os.listdir(adapter_dir))

with open('/kaggle/working/adapter_path.txt', 'w') as f:
    f.write(adapter_dir)

## Step 3: Streamed 8-bit Sharded Merge (Disk-Safe)

In [ ]:
# Write isolated merge script that saves in max 1.5GB shards to stay strictly under Kaggle disk limits
merge_script = '''
import os
os.environ["USE_TF"] = "0"
os.environ["USE_TORCH"] = "1"
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"

import peft.import_utils
peft.import_utils.is_torchao_available = lambda: False

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

with open("/kaggle/working/adapter_path.txt", "r") as f:
    adapter_dir = f.read().strip()

base_model_name = "unsloth/Meta-Llama-3.1-8B-Instruct"
merged_output_dir = "/kaggle/working/legalmitra_merged_shards"

print(f"Loading base model: {base_model_name}...")
tokenizer = AutoTokenizer.from_pretrained(base_model_name)
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    torch_dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True,
)

print(f"Applying LegalMitra LoRA adapter from: {adapter_dir}...")
model = PeftModel.from_pretrained(base_model, adapter_dir)

print("Merging LoRA weights directly into base model...")
merged_model = model.merge_and_unload()

print(f"Saving merged weights in compact shards to: {merged_output_dir}...")
merged_model.save_pretrained(merged_output_dir, max_shard_size="1800MB", safe_serialization=True)
tokenizer.save_pretrained(merged_output_dir)
print("SUCCESS: Model successfully merged into compact shards.")
'''

with open('/kaggle/working/merge_runner.py', 'w') as f:
    f.write(merge_script)

!python3 /kaggle/working/merge_runner.py

# Clear HuggingFace download cache to recover ~15 GB disk space before GGUF conversion!
!rm -rf /root/.cache/huggingface
print("SUCCESS: HuggingFace cache purged. Disk space recovered for quantization.")

## Step 4: Convert Directly to 4-bit Quantized GGUF

In [ ]:
import os
import shutil

fp16_gguf_path = "/kaggle/working/temp_fp16.gguf"
q4_gguf_path = "/kaggle/working/legalmitra-8b-q4_k_m.gguf"
quantize_bin = "/kaggle/working/llama.cpp/build/bin/llama-quantize"

print("Step 4A: Converting sharded model to intermediate FP16 GGUF...")
!python3 /kaggle/working/llama.cpp/convert_hf_to_gguf.py \
    /kaggle/working/legalmitra_merged_shards \
    --outfile {fp16_gguf_path} \
    --outtype f16

# Immediately delete the raw HF shards directory to free up ~16GB disk space before quantization!
print("Purging uncompressed HF shards to maximize disk for final quantization...")
!rm -rf /kaggle/working/legalmitra_merged_shards

print("Step 4B: Quantizing to q4_k_m (4-bit Medium)...")
!{quantize_bin} {fp16_gguf_path} {q4_gguf_path} q4_k_m

# Remove intermediate FP16 GGUF file
if os.path.exists(fp16_gguf_path):
    os.remove(fp16_gguf_path)
if os.path.exists('/kaggle/working/merge_runner.py'):
    os.remove('/kaggle/working/merge_runner.py')

q4_size_gb = os.path.getsize(q4_gguf_path) / (1024 ** 3)
print(f"\n====================================================================")
print(f"SUCCESS: Standalone GGUF model created: {q4_gguf_path}")
print(f"Final 4-bit Quantized Size: {q4_size_gb:.2f} GB")
print(f"====================================================================")
print("You can now download 'legalmitra-8b-q4_k_m.gguf' from the Output tab on the right panel!")

## Step 5: Quick Inference Test (Sanity Check)

In [ ]:
test_prompt = "<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\nWhat happens in case of minor theft under the Bharatiya Nyaya Sanhita, 2023?<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n"

cli_bin = "/kaggle/working/llama.cpp/build/bin/llama-cli"

!{cli_bin} \
    -m /kaggle/working/legalmitra-8b-q4_k_m.gguf \
    -p "{test_prompt}" \
    -n 128 \
    --temp 0.2